# Deliverable 3 - INF201: Mini-RPG battle simulator

**Group:** Theodor Furuseth and Didrik Wang Berthinsen  
**GitHub repository:** https://github.com/Didrikwang/RPG_Battle_Simulator

This notebook presents the classes in our battle simulator and runs simulated battles between different types of characters, with and without weapons.

Project structure:

| Module | Contents |
|---|---|
| `dice.py` | `Dice` |
| `items.py` | `Weapon` |
| `character.py` | `Character`, `Warrior`, `Mage` |
| `battle.py` | `Battle` |

In [17]:
import random

from dice import Dice
from items import Weapon
from character import Character, Warrior, Mage
from battle import Battle

random.seed(201)  # Fixed seed so the results in this notebook are reproducible

## 1. The `Character` class (Task 1)

**Attributes:** `name`, `_hp` (protected), `_hp_max` (protected), `damage`, `weapon`  
**Properties (read-only):** `hp`, `hp_max`, `is_alive`  
**Methods:** `take_damage()`, `heal()`, `attack()`, `roll_attack_damage()`, `introduce()`, `__repr__()`, `__str__()`

In lecture 8, `attack()` changed the target's protected `_hp` directly. Now `attack()` calls `target.take_damage()`, so each character is the only one that changes its own HP. `take_damage()` keeps HP at or above 0 and `heal()` keeps HP at or below `hp_max`. A dead character cannot be healed.

In [18]:
hero = Character("Aria")
hero.introduce()
print(hero)
hero

Hi, my name is Aria!
Character: Aria with 100/100 HP and 10 base damage


<Character name=Aria hp=100 damage=10 weapon=None>

In [19]:
print("Damage taken:", hero.take_damage(30), "-> HP:", hero.hp)
print("Healed:", hero.heal(50), "-> HP:", hero.hp, "(capped at hp_max)")
print("Damage taken:", hero.take_damage(500), "-> HP:", hero.hp, "(cannot go below 0)")
print("Is alive:", hero.is_alive)
print("Healed while dead:", hero.heal(20))

Damage taken: 30 -> HP: 70
Healed: 30 -> HP: 100 (capped at hp_max)
Damage taken: 100 -> HP: 0 (cannot go below 0)
Is alive: False
Healed while dead: 0


In [20]:
# hp and is_alive are read-only properties
try:
    hero.hp = 100
except AttributeError as error:
    print("AttributeError:", error)

AttributeError: can't set attribute 'hp'


## 2. The `Dice` class (Task 2)

**Attributes:** `_sides` (protected), read through the property `sides`  
**Methods:** `roll()`, `__call__()`, `__repr__()`

A dice must have at least 2 sides (default 6). `__call__` makes the object callable, so `d20()` does the same as `d20.roll()`.

In [21]:
d6 = Dice()
d20 = Dice(sides=20)
print(d6, d20)
print("d20.roll():", d20.roll())
print("d20():", d20())

try:
    Dice(sides=1)
except ValueError as error:
    print("ValueError:", error)

Dice(sides=6) Dice(sides=20)
d20.roll(): 3
d20(): 11
ValueError: A dice must have an integer number of sides >= 2


In [22]:
# Check that a d6 is roughly uniform over many rolls
from collections import Counter

counts = Counter(d6() for _ in range(60_000))
print(sorted(counts.items()))

[(1, 10166), (2, 9999), (3, 9972), (4, 9956), (5, 9903), (6, 10004)]


## 3. The `Weapon` class (Task 3)

**Attributes:** `name`, `damage_dice: Dice`  
**Methods:** `roll_damage()`, `__repr__()`, `__str__()`

`Weapon` uses composition: a weapon *has a* dice, and a character *has a* weapon. If a character has a weapon equipped, `attack()` rolls weapon damage instead of using base damage.

In [23]:
sword = Weapon("Sword", Dice(12))
print(sword)
sword

Sword (d12)


Weapon(name='Sword', damage_dice=Dice(sides=12))

In [24]:
knight = Character("Knight", weapon=sword)
dummy = Character("Training dummy", hp_max=1000)
print(knight)
print("Weapon damage per hit:", [knight.attack(dummy) for _ in range(8)])

Character: Knight with 100/100 HP and 10 base damage wielding Sword (d12)
Weapon damage per hit: [11, 7, 12, 12, 12, 5, 6, 5]


## 4. Subclasses: `Warrior` and `Mage` (Task 4)

Both inherit from `Character` ("is-a" relationship) and call `super().__init__()`.

**Warrior** adds `armor` (default 5) and overrides `take_damage()`: incoming damage is reduced by `armor` (never below 0) before `super().take_damage()` is called.

**Mage** adds `_mana` and `_mana_max` (read through the properties `mana` and `mana_max`), both default 30, and overrides `attack()`:
- Mana >= 10: spend 10 mana and cast a spell for double base damage. The weapon is ignored.
- Mana < 10: normal attack through `super().attack()` (uses the weapon if equipped), then restore 5 mana.

In [25]:
warrior = Warrior("Conan")
print(warrior)
print("Hit for 3, HP lost:", warrior.take_damage(3))
print("Hit for 12, HP lost:", warrior.take_damage(12))
print("HP now:", warrior.hp)

Warrior: Conan with 100/100 HP and 10 base damage, armor 5
Hit for 3, HP lost: 0
Hit for 12, HP lost: 7
HP now: 93


In [26]:
mage = Mage("Merlin", weapon=Weapon("Staff", Dice(4)))
target = Character("Target", hp_max=1000)
for attack_number in range(1, 9):
    damage = mage.attack(target)
    print(f"Attack {attack_number}: {damage:>2} damage, mana left: {mage.mana}")

Attack 1: 20 damage, mana left: 20
Attack 2: 20 damage, mana left: 10
Attack 3: 20 damage, mana left: 0
Attack 4:  2 damage, mana left: 5
Attack 5:  4 damage, mana left: 10
Attack 6: 20 damage, mana left: 0
Attack 7:  2 damage, mana left: 5
Attack 8:  1 damage, mana left: 10


The first three attacks are spells (20 damage each) until mana drops below 10. After that the mage alternates between staff attacks (which restore 5 mana) and spells.

In [27]:
print(isinstance(warrior, Character), isinstance(mage, Character))
print(Mage.__mro__)

True True
(<class 'character.Mage'>, <class 'character.Character'>, <class 'object'>)


## 5. The `Battle` class (Task 5)

**Attributes:** `hero`, `enemy`, `round_number`, `max_rounds`, `verbose`  
**Property:** `winner`  
**Methods:** `play_round()`, `simulate_battle()`

Each round the hero attacks first, and the enemy only strikes back if it is still alive. `simulate_battle()` runs rounds in a `while` loop until one character is dead and returns the winner. We added `max_rounds` as a safety limit: two warriors whose damage is lower than the opponent's armor can never hurt each other, and without the limit the loop would run forever.

### 5.1 Two basic characters, no weapons
Fixed damage and no randomness, so the hero always wins by attacking first.

In [28]:
Battle(Character("Hero"), Character("Enemy")).simulate_battle()

BATTLE: Hero vs Enemy
Round 1
  Hero hits Enemy for 10 damage -> Enemy: 90/100 HP
  Enemy hits Hero for 10 damage -> Hero: 90/100 HP
Round 2
  Hero hits Enemy for 10 damage -> Enemy: 80/100 HP
  Enemy hits Hero for 10 damage -> Hero: 80/100 HP
Round 3
  Hero hits Enemy for 10 damage -> Enemy: 70/100 HP
  Enemy hits Hero for 10 damage -> Hero: 70/100 HP
Round 4
  Hero hits Enemy for 10 damage -> Enemy: 60/100 HP
  Enemy hits Hero for 10 damage -> Hero: 60/100 HP
Round 5
  Hero hits Enemy for 10 damage -> Enemy: 50/100 HP
  Enemy hits Hero for 10 damage -> Hero: 50/100 HP
Round 6
  Hero hits Enemy for 10 damage -> Enemy: 40/100 HP
  Enemy hits Hero for 10 damage -> Hero: 40/100 HP
Round 7
  Hero hits Enemy for 10 damage -> Enemy: 30/100 HP
  Enemy hits Hero for 10 damage -> Hero: 30/100 HP
Round 8
  Hero hits Enemy for 10 damage -> Enemy: 20/100 HP
  Enemy hits Hero for 10 damage -> Hero: 20/100 HP
Round 9
  Hero hits Enemy for 10 damage -> Enemy: 10/100 HP
  Enemy hits Hero for 10 damag

<Character name=Hero hp=10 damage=10 weapon=None>

### 5.2 Characters with weapons

In [29]:
battle = Battle(Character("Axe fighter", weapon=Weapon("Axe", Dice(20))),
                Character("Swordsman", weapon=Weapon("Sword", Dice(12))))
winner = battle.simulate_battle()

BATTLE: Axe fighter vs Swordsman
Round 1
  Axe fighter hits Swordsman for 18 damage -> Swordsman: 82/100 HP
  Swordsman hits Axe fighter for 8 damage -> Axe fighter: 92/100 HP
Round 2
  Axe fighter hits Swordsman for 6 damage -> Swordsman: 76/100 HP
  Swordsman hits Axe fighter for 3 damage -> Axe fighter: 89/100 HP
Round 3
  Axe fighter hits Swordsman for 12 damage -> Swordsman: 64/100 HP
  Swordsman hits Axe fighter for 11 damage -> Axe fighter: 78/100 HP
Round 4
  Axe fighter hits Swordsman for 17 damage -> Swordsman: 47/100 HP
  Swordsman hits Axe fighter for 1 damage -> Axe fighter: 77/100 HP
Round 5
  Axe fighter hits Swordsman for 12 damage -> Swordsman: 35/100 HP
  Swordsman hits Axe fighter for 12 damage -> Axe fighter: 65/100 HP
Round 6
  Axe fighter hits Swordsman for 5 damage -> Swordsman: 30/100 HP
  Swordsman hits Axe fighter for 1 damage -> Axe fighter: 64/100 HP
Round 7
  Axe fighter hits Swordsman for 5 damage -> Swordsman: 25/100 HP
  Swordsman hits Axe fighter for 7 

### 5.3 Warrior vs Mage

In [30]:
battle = Battle(Warrior("Conan", weapon=Weapon("Sword", Dice(12))),
                Mage("Merlin", weapon=Weapon("Staff", Dice(6))))
winner = battle.simulate_battle()
print(winner)

BATTLE: Conan vs Merlin
Round 1
  Conan hits Merlin for 1 damage -> Merlin: 99/100 HP
  Merlin hits Conan for 15 damage -> Conan: 85/100 HP
Round 2
  Conan hits Merlin for 6 damage -> Merlin: 93/100 HP
  Merlin hits Conan for 15 damage -> Conan: 70/100 HP
Round 3
  Conan hits Merlin for 2 damage -> Merlin: 91/100 HP
  Merlin hits Conan for 15 damage -> Conan: 55/100 HP
Round 4
  Conan hits Merlin for 3 damage -> Merlin: 88/100 HP
  Merlin hits Conan for 1 damage -> Conan: 54/100 HP
Round 5
  Conan hits Merlin for 6 damage -> Merlin: 82/100 HP
  Merlin hits Conan for 0 damage -> Conan: 54/100 HP
Round 6
  Conan hits Merlin for 8 damage -> Merlin: 74/100 HP
  Merlin hits Conan for 15 damage -> Conan: 39/100 HP
Round 7
  Conan hits Merlin for 1 damage -> Merlin: 73/100 HP
  Merlin hits Conan for 0 damage -> Conan: 39/100 HP
Round 8
  Conan hits Merlin for 3 damage -> Merlin: 70/100 HP
  Merlin hits Conan for 0 damage -> Conan: 39/100 HP
Round 9
  Conan hits Merlin for 7 damage -> Merlin: 

### 5.4 Warrior vs Warrior where armor blocks everything (draw)

In [31]:
battle = Battle(Warrior("Tank A", damage=4), Warrior("Tank B", damage=4), max_rounds=20, verbose=False)
print("Winner:", battle.simulate_battle(), "after", battle.round_number, "rounds")

Winner: None after 20 rounds


### 5.5 Many simulated battles
One battle says little when damage is random. We run each matchup 1000 times with fresh characters and count wins. Each matchup is run in both orders so the first-strike advantage evens out.

In [32]:
def win_rates(make_a, make_b, n_battles: int = 1000) -> dict:
    """Run n_battles in each order and return win counts by character type."""
    results = Counter()
    for _ in range(n_battles):
        for hero, enemy in ((make_a(), make_b()), (make_b(), make_a())):
            winner = Battle(hero, enemy, verbose=False).simulate_battle()
            results[winner.name if winner else "Draw"] += 1
    return dict(results)

matchups = {
    "Warrior vs Mage (no weapons)": (lambda: Warrior("Warrior"), lambda: Mage("Mage")),
    "Warrior (d12 sword) vs Mage (d6 staff)": (
        lambda: Warrior("Warrior", weapon=Weapon("Sword", Dice(12))),
        lambda: Mage("Mage", weapon=Weapon("Staff", Dice(6)))),
    "Character (d20 axe) vs Warrior": (
        lambda: Character("Character", weapon=Weapon("Axe", Dice(20))), lambda: Warrior("Warrior")),
    "Character (d20 axe) vs Mage": (
        lambda: Character("Character", weapon=Weapon("Axe", Dice(20))), lambda: Mage("Mage")),
}

for title, (make_a, make_b) in matchups.items():
    print(f"{title:<42} {win_rates(make_a, make_b)}")

Warrior vs Mage (no weapons)               {'Warrior': 1000, 'Mage': 1000}
Warrior (d12 sword) vs Mage (d6 staff)     {'Warrior': 723, 'Mage': 1277}
Character (d20 axe) vs Warrior             {'Warrior': 1993, 'Character': 7}
Character (d20 axe) vs Mage                {'Mage': 1996, 'Character': 4}


## 6. Discussion
Results

Warrior vs Mage without weapons: fully deterministic. Both need exactly 10 rounds to kill the other, so whoever attacks first always wins.

Warrior (d12 sword) vs Mage (d6 staff): the mage wins about 64 %. The sword averages 6.5, lower than the warrior’s base damage of 10, so the weapon makes the warrior weaker. The mage’s spells ignore its weapon and still deal 15 through armor.

Basic character with a d20 axe: loses over 99 % against both specialisations. Armor removes rolls 1 to 5, cutting the axe’s average from 10.5 to 6.0, and the mage’s opening spell burst (60 damage in three rounds) is too strong.

Design choices

Encapsulation: _hp and _mana are only changed by the character’s own methods. hp, mana and is_alive are read-only properties, which fixes the lecture 8 problem of attack() editing another object’s _hp.

Composition and inheritance: a Weapon has a Dice, a Character has a Weapon. Warrior and Mage are characters and only override what makes them special (take_damage() and attack()).

Polymorphism: Battle only uses attack() and is_alive, so it works with any character type.

Safety: max_rounds prevents infinite loops when no one can deal damage, and invalid input raises ValueError.

Limitations

The hero always attacks first. An initiative roll would be fairer.

Weapons replace base damage instead of adding to it, so a weak weapon can make a character worse.
heal() is never used in battles. Healing potions could be added to items.py.


## 7. Use of GenAI

## Use of generative AI

We used Claude (Anthropic) as a sparring partner while solving this deliverable. We wrote our own versions of all classes and notebook cells, and used the AI to review them, explain concepts and test the code. All final decisions were ours.

### How we used it

**Task 1: Character**
- The AI drafted `take_damage()`, `heal()` and `is_alive`. We wrote our own version with return values (HP actually lost or restored), a `roll_attack_damage()` helper and a rule that dead characters cannot be healed.
- The AI pointed out that `hp_max` could still be changed from outside the class (e.g. `c.hp_max = 10` while HP is 100), and suggested making it a read-only property.
- When we asked about the difference between `self.mana_max` and `self._mana_max`, the AI explained that the property is a read-only view of the protected attribute. It also corrected its own earlier advice: only the assignments in `__init__` had to change, while reading the value through the property works fine.

**Task 2: Dice**
- The AI drafted the class and noted that the assignment text says `d20.roll()` when describing `__call__`, while `__call__` actually enables `d20()`. Our version added an `isinstance` check. The AI suggested raising `TypeError` for wrong types as an optional improvement.

**Task 3: Weapon**
- The AI drafted the class, and our version was essentially the same. It suggested an optional type check for `damage_dice`.

**Task 4: Warrior and Mage**
- The AI drafted both subclasses. We adapted them with our own `__str__` methods. The AI suggested making `mana_max` read-only to match `mana`.

**Task 5: Battle**
- The AI found that a battle can loop forever: two warriors with d4 daggers dealt 0 damage in 1000 hits, because armor 5 blocks every roll. This led to the `max_rounds` limit.
- We used our own Battle class (with a start-of-battle alive check and a `winner` property) instead of the AI's draft. The AI suggested validating `max_rounds >= 1`.

**Notebook**
- We wrote the cells. The AI suggested additions: demonstrating `attack()`, testing all read-only properties, contrasting armed and unarmed attacks, showing polymorphism (a plain Character attacking a Warrior still triggers armor), using `random.seed()` for reproducible output and explaining MRO.
- Our `win_rates()` function runs every matchup in both turn orders. This showed that two of the AI's earlier claims were wrong: "Warrior without weapon always wins" and "Warrior (d12) vs Mage (d6) is about 50/50" were both caused by the Warrior always attacking first. The AI acknowledged and corrected this.
- The AI checked the numbers in our discussion (10 rounds each way, an average of 6.0 for d20 minus armor, 60 spell damage in three rounds) and suggested wording and formatting fixes.

### Reflection
- The AI was most useful for reviewing code, finding edge cases (the infinite loop, mutable `hp_max`) and explaining concepts.
- It also made mistakes. Its first simulation results were biased by turn order, which we only discovered by designing a fairer test ourselves. AI output therefore has to be verified, not trusted blindly.
- All AI-suggested code was read, tested and adapted by us before use.